# Engine benchmark: five variants, three workloadsEach variant answers each of three identical questions. Variant is the storageand execution path; workload is the question.| Variant | Execution | Companies source | Financial source ||---|---|---|---|| A1 | `mongod`, server-side C++ | MongoDB | MongoDB || A2 | `mongod`, server-side C++ | MongoDB | MongoDB || B | Spark JVM, local mode | MongoDB via connector | MongoDB via connector || C | Spark JVM, local mode | Parquet | Parquet || D | Spark JVM, local mode | raw `enheter_alle.json` | NDJSON export |A1 and A2 differ only in how the aggregation is written — same engine, samedata, same hardware, same session. **Query formulation is an experimentaldimension in its own right here**, because measurement showed it outweighingengine choice on W1, and because a version of this notebook that carried onlyone MongoDB formulation produced a 6.4x apparent regression that was entirelycaused by the formulation changing between runs.**A1 drives from `financial_data`.** Scans all 1,170,290 financial documents,`$lookup`s each into `companies` through `organisasjonsnummer_1`, then discardsthe roughly 739K non-AS rows. Filter after join.**A2 drives from `companies`.** Matches AS first through`organisasjonsform.kode_1`, cutting the driving set to 431,581, then `$lookup`sinto `financial_data` on `_id`. Since `financial_data._id` *is* theorganisasjonsnummer, this probes the primary index. Filter before join.Both must return identical results; the correctness check verifies that ratherthan assuming it. If they ever disagree, their timings are not comparable andthe formulation figure is meaningless.Variant A is not a Python join. pymongo sends the pipeline to `mongod`, whichexecutes it; Python only deserialises the result.**Variant D is asymmetric by design.** `enheter_alle.json` is a singlepretty-printed JSON array of 2.00 GB. Spark can read it only with`multiLine=true`, which is not splittable: one task parses the whole file on onethread regardless of `local[4]`. The financial side is NDJSON and reads inparallel. This mirrors the real difference between a bulk download and anincremental fetch, and the per-side read timings separate the two effects.## Workloads**W1 — selective join, two output rows.** Join financial data to companies onorganisation number, keep AS, count by `fetch_status`.**W3 — unindexed predicate, no join.** Count `konkurs = true` grouped by legalform. No index on `konkurs`, so A scans the whole collection. Single collection,so there is no join to reorder and only one MongoDB formulation exists.**W4 — wide read plus real aggregation.** Operating revenue, operating profit,equity and debt for AS companies with a filed statement, grouped by industrycode and municipality. Reads deeply nested columns from both sources andproduces tens of thousands of groups. Output feeds the analysis chapter.## Ordering**Read-only timings run last.** They pull 2.00 GB of JSON plus the Parquet andNDJSON exports through the page cache before anything else executes, which couldin principle penalise a MongoDB variant running afterwards. Measurement found nosuch effect — `Diagnose_variant_a.ipynb` recorded zero bytes read into theWiredTiger cache and zero page evictions across a full isolated run — so this isprecautionary, not corrective. `timeit` warms each variant individually, so novariant depends on an earlier one having warmed the cache.## CorrectnessEvery workload is checked across all variants that completed, against the firstof them as reference rather than a fixed variant. A hardcoded reference meansthat if that one variant fails, `compare(None, res)` returns False and everyother variant is reported as disagreeing when it was never compared to anything.Two engine semantics differ and are reconciled in the query rather than afterit. `$ifNull` forces an explicit null for a missing group key, because MongoDBotherwise omits the key entirely while Spark carries a null — same absence, tworepresentations. And Spark's `SUM` over zero non-null values is null whileMongoDB's `$sum` is 0, so the Spark aggregate coalesces to `0.0`, adoptingMongoDB's convention explicitly. Both cases occur: `naeringskode1` is absent in35,727 companies, `kommunenummer` in 62,388, `sumDriftsinntekter` in 91,026filings.Run `Export_to_parquet.ipynb` and `Export_to_ndjson.ipynb` first. Restart thekernel before running this.

In [1]:
import json
import os
import statistics
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from schemas import COMPANIES_SCHEMA, FINANCIAL_SCHEMA

# Row counts from the snapshot dated 2026-08-25, used to fail loudly if the
# collections have changed under the benchmark. Defined here rather than
# imported from schemas.py, because the notebook should not break when that
# file is edited. If you prefer it to live in schemas.py, import it instead and
# delete this.
EXPECTED_ROWS = {"companies": 1171373, "financial_data": 1170290}

MONGO_DB = "companiesdb"
DATA_DIR = "/home/jovyan/data"
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")
NDJSON_DIR = os.path.join(DATA_DIR, "ndjson")
RAW_COMPANIES = os.path.join(DATA_DIR, "enheter_alle.json")

REPEATS = 3          # timed runs per cell, after one discarded warm-up

# Restrict to a subset to run variants in separate kernels.
SELECTED_VARIANTS = ["A1", "A2", "B", "C", "D"]

# Driver memory, thread count, the Mongo connector package and the connection
# URI all come from jupyter/spark-defaults.conf, which is baked into the image.
spark = SparkSession.builder.appName("group13_engine_benchmark").getOrCreate()

_conf = spark.sparkContext.getConf()
MONGO_URI = _conf.get("spark.mongodb.read.connection.uri")
CONNECTOR = _conf.get("spark.jars.packages")

print("Spark        ", spark.version)
print("master       ", spark.sparkContext.master)
print("driver heap  %.1f GB" % (spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3))
print("connector    ", CONNECTOR)
print("host cores   ", os.cpu_count())
print("variants     ", SELECTED_VARIANTS)

Spark         4.2.0
master        local[4]
driver heap  8.0 GB
connector     org.mongodb.spark:mongo-spark-connector_2.13:11.1.0
host cores    12
variants      ['A1', 'A2', 'B', 'C', 'D']


In [2]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

# These figures must appear in the report. A timing without the hardware and
# data state it ran on is not reproducible.
with open("/proc/meminfo") as fh:
    total_kb = int(fh.readline().split()[1])

print("Container memory:    %.1f GB" % (total_kb / 1024**2))
print("Default parallelism: ", spark.sparkContext.defaultParallelism)
print("MongoDB version:     ", db.command("buildInfo")["version"])
print("Raw JSON size:       %.2f GB" % (os.path.getsize(RAW_COMPANIES) / 1024**3))

# Recorded because they are the first things to check when a timing moves
# between runs, and because which formulations are index-served depends on them.
COLLECTION_COUNTS = {}
for name in ["companies", "financial_data"]:
    n = db[name].count_documents({})
    COLLECTION_COUNTS[name] = n
    print("%-16s %d docs  %s" % (name, n, "OK" if n == EXPECTED_ROWS[name] else "CHANGED"))

INDEXES = {n: {i: s.get("key") for i, s in db[n].index_information().items()}
           for n in ["companies", "financial_data"]}
print()
for name, idx in INDEXES.items():
    print("%-16s %s" % (name, list(idx)))


def timeit(fn, repeats=REPEATS):
    """
    Runs fn once untimed so the page cache is warm and the JVM has JIT-compiled
    the hot path, then times `repeats` further runs. Returns the last result and
    the timings. Median is reported rather than mean, so a single GC pause or
    Docker scheduling hiccup does not dominate.

    Returns (result, times, error). On failure the error is recorded and the
    variant is reported as not completing, rather than aborting the notebook.
    """
    try:
        result = fn()
        times = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            result = fn()
            times.append(time.perf_counter() - t0)
        return result, times, None
    except Exception as exc:                       # noqa: BLE001
        return None, [], "%s: %s" % (type(exc).__name__, exc)

Container memory:    15.2 GB
Default parallelism:  4
MongoDB version:      8.3.8
Raw JSON size:       1.86 GB
companies        1171373 docs  OK
financial_data   1170291 docs  CHANGED

companies        ['_id_', 'organisasjonsnummer_1', 'organisasjonsform.kode_1']
financial_data   ['_id_']


## Source readersEach variant's readers are defined once. Nothing is cached: every timed runre-reads from its source, which is the cost being measured.

In [3]:
def mongo_df(collection, schema):
    return (spark.read.format("mongodb")
            .option("database", MONGO_DB).option("collection", collection)
            .schema(schema).load())


def parquet_df(collection):
    return spark.read.parquet(os.path.join(PARQUET_DIR, collection))


def json_companies():
    # multiLine is mandatory: the file is one array, not one object per line.
    # It also makes the read single-threaded, which is the point of variant D.
    return (spark.read.schema(COMPANIES_SCHEMA)
            .option("multiLine", "true").json(RAW_COMPANIES))


def json_financial():
    return (spark.read.schema(FINANCIAL_SCHEMA)
            .json(os.path.join(NDJSON_DIR, "financial_data")))


ALL_SOURCES = {
    "B: Spark + Mongo connector": (lambda: mongo_df("companies", COMPANIES_SCHEMA),
                                   lambda: mongo_df("financial_data", FINANCIAL_SCHEMA)),
    "C: Spark + Parquet":         (lambda: parquet_df("companies"),
                                   lambda: parquet_df("financial_data")),
    "D: Spark + JSON":            (json_companies, json_financial),
}

SOURCES = {k: v for k, v in ALL_SOURCES.items() if k.split(":")[0] in SELECTED_VARIANTS}
RUN_A1 = "A1" in SELECTED_VARIANTS
RUN_A2 = "A2" in SELECTED_VARIANTS
print("Spark variants:", list(SOURCES))
print("MongoDB formulations: A1=%s  A2=%s" % (RUN_A1, RUN_A2))

Spark variants: ['B: Spark + Mongo connector', 'C: Spark + Parquet', 'D: Spark + JSON']
MongoDB formulations: A1=True  A2=True


## Result comparisonGroup keys and integer counts must match exactly across variants. Sums offloating-point columns are compared with a relative tolerance of 1e-9, because`mongod` and Spark accumulate in different orders.

In [4]:
TOLERANCE = 1e-9


def normalise(rows):
    """
    rows: iterable of (key_tuple, int_count, tuple_of_floats).
    Sorted by key so variants can be compared elementwise.
    """
    out = []
    for key, count, floats in rows:
        key = tuple("" if k is None else str(k) for k in key)
        floats = tuple(None if f is None else float(f) for f in floats)
        out.append((key, int(count), floats))
    return sorted(out)


def compare(a, b):
    """True if two normalised results agree within tolerance."""
    if a is None or b is None or len(a) != len(b):
        return False
    for (ka, ca, fa), (kb, cb, fb) in zip(a, b):
        if ka != kb or ca != cb or len(fa) != len(fb):
            return False
        for x, y in zip(fa, fb):
            if (x is None) != (y is None):
                return False
            if x is None:
                continue
            scale = max(abs(x), abs(y), 1.0)
            if abs(x - y) / scale > TOLERANCE:
                return False
    return True


def report(wdata):
    for label, (res, times, err) in wdata.items():
        if err:
            print("%-32s DID NOT COMPLETE  %s" % (label, err))
        else:
            print("%-32s median %6.2fs   min %6.2fs   max %6.2fs   %d rows"
                  % (label, statistics.median(times), min(times), max(times), len(res)))

## W1 — selective join, two output rowsA1 drives from `financial_data`: 1,170,290 lookups, filter after join.A2 drives from `companies`: index-served AS match first, 431,581 lookups against`financial_data._id`, filter before join.

In [5]:
# A1: filter after join. Scans all financial documents.
W1_A1 = [
    {"$lookup": {"from": "companies", "localField": "organisasjonsnummer",
                 "foreignField": "organisasjonsnummer", "as": "company"}},
    {"$unwind": "$company"},
    {"$match": {"company.organisasjonsform.kode": "AS"}},
    {"$group": {"_id": "$fetch_status", "count": {"$sum": 1}}},
]

# A2: filter before join, via organisasjonsform.kode_1, then probe
# financial_data._id, which is the organisasjonsnummer and therefore the
# primary index.
W1_A2 = [
    {"$match": {"organisasjonsform.kode": "AS"}},
    {"$lookup": {"from": "financial_data", "localField": "organisasjonsnummer",
                 "foreignField": "_id", "as": "fin"}},
    {"$unwind": "$fin"},
    {"$group": {"_id": "$fin.fetch_status", "count": {"$sum": 1}}},
]


def w1_a1():
    rows = db.financial_data.aggregate(W1_A1, allowDiskUse=True)
    return normalise(((r["_id"],), r["count"], ()) for r in rows)


def w1_a2():
    rows = db.companies.aggregate(W1_A2, allowDiskUse=True)
    return normalise(((r["_id"],), r["count"], ()) for r in rows)


def w1_spark(comp_fn, fin_fn):
    companies = comp_fn().filter("organisasjonsform.kode = 'AS'").select("organisasjonsnummer")
    financial = fin_fn().select("organisasjonsnummer", "fetch_status")
    rows = (financial.join(companies, "organisasjonsnummer")
            .groupBy("fetch_status").count().collect())
    return normalise(((r["fetch_status"],), r["count"], ()) for r in rows)


w1 = {}
if RUN_A1:
    w1["A1: MongoDB (financial-first)"] = timeit(w1_a1)
if RUN_A2:
    w1["A2: MongoDB (AS-first)"] = timeit(w1_a2)
for label, (comp_fn, fin_fn) in SOURCES.items():
    w1[label] = timeit(lambda c=comp_fn, f=fin_fn: w1_spark(c, f))

report(w1)

A1: MongoDB (financial-first)    median  38.82s   min  38.73s   max  39.04s   2 rows
A2: MongoDB (AS-first)           median   6.94s   min   6.81s   max   6.95s   2 rows
B: Spark + Mongo connector       median   9.89s   min   9.24s   max  10.57s   2 rows
C: Spark + Parquet               median   2.62s   min   2.54s   max   2.64s   2 rows
D: Spark + JSON                  median  36.95s   min  36.44s   max  37.70s   2 rows


## W3 — unindexed predicate, no join`konkurs` carries no index, so variant A scans the full collection here ratherthan seeking through an index as it did in W1. Single collection, so there is nojoin to reorder and only one MongoDB formulation exists.

In [6]:
W3_PIPELINE = [
    {"$match": {"konkurs": True}},
    {"$group": {"_id": "$organisasjonsform.kode", "count": {"$sum": 1}}},
]


def w3_mongo():
    rows = db.companies.aggregate(W3_PIPELINE, allowDiskUse=True)
    return normalise(((r["_id"],), r["count"], ()) for r in rows)


def w3_spark(comp_fn):
    rows = (comp_fn().filter(F.col("konkurs"))
            .groupBy(F.col("organisasjonsform.kode").alias("kode"))
            .count().collect())
    return normalise(((r["kode"],), r["count"], ()) for r in rows)


w3 = {}
if RUN_A1 or RUN_A2:
    w3["A: MongoDB aggregation"] = timeit(w3_mongo)
for label, (comp_fn, _) in SOURCES.items():
    w3[label] = timeit(lambda c=comp_fn: w3_spark(c))

report(w3)

A: MongoDB aggregation           median   0.38s   min   0.36s   max   0.40s   11 rows
B: Spark + Mongo connector       median   1.40s   min   1.37s   max   1.44s   11 rows
C: Spark + Parquet               median   1.55s   min   1.30s   max   1.62s   11 rows
D: Spark + JSON                  median  34.15s   min  32.81s   max  34.64s   11 rows


## W4 — wide read plus real aggregationOperating revenue, operating profit, equity and debt for AS companies with afiled statement, grouped by industry code and municipality.`data` is typed as an array because that is what the API returns, but theprofiling pass confirmed exactly one element in all 444,644 populated records,so element 0 is taken directly in Spark and the join stays one-to-one. `$unwind`is equivalent at multiplicity 1.As in W1, A1 filters after the join and A2 filters before it.

In [7]:
W4_SUMS = ["inntekt", "driftsres", "egenkapital", "gjeld"]

# A1: drive from financial_data, filter AS after the join.
W4_A1 = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$lookup": {"from": "companies", "localField": "organisasjonsnummer",
                 "foreignField": "organisasjonsnummer", "as": "c"}},
    {"$unwind": "$c"},
    {"$match": {"c.organisasjonsform.kode": "AS"}},
    {"$group": {
        # $ifNull forces an explicit null. Without it MongoDB omits a key whose
        # value is missing, so a company with no industry code yields an _id
        # with no "naering" field at all, while Spark yields a null. Same
        # absence, two representations; this makes them comparable.
        "_id": {"naering": {"$ifNull": ["$c.naeringskode1.kode", None]},
                "kommune": {"$ifNull": ["$c.forretningsadresse.kommunenummer", None]}},
        "n": {"$sum": 1},
        "inntekt": {"$sum": "$data.resultatregnskapResultat.driftsresultat.driftsinntekter.sumDriftsinntekter"},
        "driftsres": {"$sum": "$data.resultatregnskapResultat.driftsresultat.driftsresultat"},
        "egenkapital": {"$sum": "$data.egenkapitalGjeld.egenkapital.sumEgenkapital"},
        "gjeld": {"$sum": "$data.egenkapitalGjeld.gjeldOversikt.sumGjeld"},
    }},
]

# A2: drive from companies, filter AS first via organisasjonsform.kode_1, then
# probe financial_data._id. Paths are written out in full rather than assembled
# from fragments, which is how an earlier version produced "$findata..." instead
# of "$fin.data..." and silently nulled every sum.
W4_A2 = [
    {"$match": {"organisasjonsform.kode": "AS"}},
    {"$lookup": {"from": "financial_data", "localField": "organisasjonsnummer",
                 "foreignField": "_id", "as": "fin"}},
    {"$unwind": "$fin"},
    {"$match": {"fin.fetch_status": "success"}},
    {"$unwind": "$fin.data"},
    {"$group": {
        "_id": {"naering": {"$ifNull": ["$naeringskode1.kode", None]},
                "kommune": {"$ifNull": ["$forretningsadresse.kommunenummer", None]}},
        "n": {"$sum": 1},
        "inntekt": {"$sum": "$fin.data.resultatregnskapResultat.driftsresultat.driftsinntekter.sumDriftsinntekter"},
        "driftsres": {"$sum": "$fin.data.resultatregnskapResultat.driftsresultat.driftsresultat"},
        "egenkapital": {"$sum": "$fin.data.egenkapitalGjeld.egenkapital.sumEgenkapital"},
        "gjeld": {"$sum": "$fin.data.egenkapitalGjeld.gjeldOversikt.sumGjeld"},
    }},
]


def _w4_rows(cursor):
    return normalise(
        ((r["_id"]["naering"], r["_id"]["kommune"]), r["n"],
         tuple(r[k] for k in W4_SUMS))
        for r in cursor)


def w4_a1():
    return _w4_rows(db.financial_data.aggregate(W4_A1, allowDiskUse=True))


def w4_a2():
    return _w4_rows(db.companies.aggregate(W4_A2, allowDiskUse=True))


def w4_spark(comp_fn, fin_fn):
    companies = comp_fn().filter("organisasjonsform.kode = 'AS'").select(
        "organisasjonsnummer",
        F.col("naeringskode1.kode").alias("naering"),
        F.col("forretningsadresse.kommunenummer").alias("kommune"))
    financial = fin_fn().filter("fetch_status = 'success'").select(
        "organisasjonsnummer", F.col("data")[0].alias("d"))
    rows = (financial.join(companies, "organisasjonsnummer")
            .groupBy("naering", "kommune")
            # Spark's SUM over zero non-null values is null; MongoDB's $sum is 0.
            # A group where every member lacks a figure would otherwise be
            # reported as a disagreement. Coalescing to 0.0 adopts MongoDB's
            # convention explicitly rather than leaving the two engines to
            # differ on an empty sum.
            .agg(F.count(F.lit(1)).alias("n"),
                 F.coalesce(F.sum("d.resultatregnskapResultat.driftsresultat.driftsinntekter.sumDriftsinntekter"), F.lit(0.0)).alias("inntekt"),
                 F.coalesce(F.sum("d.resultatregnskapResultat.driftsresultat.driftsresultat"), F.lit(0.0)).alias("driftsres"),
                 F.coalesce(F.sum("d.egenkapitalGjeld.egenkapital.sumEgenkapital"), F.lit(0.0)).alias("egenkapital"),
                 F.coalesce(F.sum("d.egenkapitalGjeld.gjeldOversikt.sumGjeld"), F.lit(0.0)).alias("gjeld"))
            .collect())
    return normalise(((r["naering"], r["kommune"]), r["n"],
                      tuple(r[k] for k in W4_SUMS)) for r in rows)


w4 = {}
if RUN_A1:
    w4["A1: MongoDB (financial-first)"] = timeit(w4_a1)
if RUN_A2:
    w4["A2: MongoDB (AS-first)"] = timeit(w4_a2)
for label, (comp_fn, fin_fn) in SOURCES.items():
    w4[label] = timeit(lambda c=comp_fn, f=fin_fn: w4_spark(c, f))

report(w4)

A1: MongoDB (financial-first)    median  18.74s   min  18.51s   max  18.75s   45033 rows
A2: MongoDB (AS-first)           median  16.42s   min  16.14s   max  16.53s   45033 rows
B: Spark + Mongo connector       median  19.68s   min  18.48s   max  21.59s   45033 rows
C: Spark + Parquet               median   3.25s   min   3.21s   max   4.30s   45033 rows
D: Spark + JSON                  median  48.59s   min  47.14s   max  48.94s   45033 rows


## CorrectnessThe reference is the first variant that completed, not a fixed one. A1 and A2must agree with each other and with every Spark variant; if they do not, the twoformulations are not answering the same question and the formulation figurebelow is meaningless.

In [8]:
WORKLOADS = [("W1", w1), ("W3", w3), ("W4", w4)]

agreement = {}
for wname, wdata in WORKLOADS:
    completed = [(l, r) for l, (r, t, e) in wdata.items() if not e]
    agreement[wname] = {}
    print("\n=== %s ===" % wname)
    if not completed:
        print("  no variant completed")
        continue
    ref_label, ref = completed[0]
    agreement[wname]["_reference"] = ref_label
    print("  reference: %s" % ref_label)
    for label, (res, times, err) in wdata.items():
        if err:
            print("  %-32s did not complete" % label)
            continue
        ok = compare(ref, res)
        agreement[wname][label] = ok
        print("  %-32s %s  (%d rows)" % (label, "agrees" if ok else "DISAGREES", len(res)))


=== W1 ===
  reference: A1: MongoDB (financial-first)
  A1: MongoDB (financial-first)    agrees  (2 rows)
  A2: MongoDB (AS-first)           agrees  (2 rows)
  B: Spark + Mongo connector       agrees  (2 rows)
  C: Spark + Parquet               agrees  (2 rows)
  D: Spark + JSON                  agrees  (2 rows)

=== W3 ===
  reference: A: MongoDB aggregation
  A: MongoDB aggregation           agrees  (11 rows)
  B: Spark + Mongo connector       agrees  (11 rows)
  C: Spark + Parquet               agrees  (11 rows)
  D: Spark + JSON                  agrees  (11 rows)

=== W4 ===
  reference: A1: MongoDB (financial-first)
  A1: MongoDB (financial-first)    agrees  (45033 rows)
  A2: MongoDB (AS-first)           agrees  (45033 rows)
  B: Spark + Mongo connector       agrees  (45033 rows)
  C: Spark + Parquet               agrees  (45033 rows)
  D: Spark + JSON                  agrees  (45033 rows)


## Formulation effect against engine effectA1 against A2 isolates how the query was written. Best MongoDB against bestSpark isolates the engine and storage format. Reporting them side by side keepsthe two from being conflated: the naive A1-against-Parquet ratio is the productof both, and quoting it alone attributes to the engine what the queryformulation caused.

In [9]:
# Names are suffixed deliberately: using a bare `spark` here would shadow the
# SparkSession and break the summary cell below.
formulation = {}
for wname, wdata in WORKLOADS:
    a1 = wdata.get("A1: MongoDB (financial-first)")
    a2 = wdata.get("A2: MongoDB (AS-first)")
    if not a1 or not a2 or a1[2] or a2[2]:
        continue
    m1, m2 = statistics.median(a1[1]), statistics.median(a2[1])
    formulation[wname] = {"a1_median": m1, "a2_median": m2, "speedup": m1 / m2}
    print("%-6s A1 %6.2fs   A2 %6.2fs   formulation effect %5.2fx" % (wname, m1, m2, m1 / m2))

print()
engine = {}
for wname, wdata in WORKLOADS:
    completed = {l: statistics.median(t) for l, (r, t, e) in wdata.items() if not e}
    mongo_times = {l: v for l, v in completed.items() if l.startswith("A")}
    spark_times = {l: v for l, v in completed.items() if not l.startswith("A")}
    if not mongo_times or not spark_times:
        continue
    best_mongo, best_spark = min(mongo_times.values()), min(spark_times.values())
    ratio = max(best_mongo, best_spark) / min(best_mongo, best_spark)
    engine[wname] = {"best_mongo": best_mongo, "best_spark": best_spark,
                     "ratio": ratio,
                     "faster": "MongoDB" if best_mongo < best_spark else "Spark"}
    print("%-6s best MongoDB %6.2fs   best Spark %6.2fs   engine effect %5.2fx (%s faster)"
          % (wname, best_mongo, best_spark, ratio, engine[wname]["faster"]))

W1     A1  38.82s   A2   6.94s   formulation effect  5.60x
W4     A1  18.74s   A2  16.42s   formulation effect  1.14x

W1     best MongoDB   6.94s   best Spark   2.62s   engine effect  2.64x (Spark faster)
W3     best MongoDB   0.38s   best Spark   1.40s   engine effect  3.70x (MongoDB faster)
W4     best MongoDB  16.42s   best Spark   3.25s   engine effect  5.06x (Spark faster)


## Read-only timingsDeliberately last, so that pulling 2.00 GB of JSON through the page cache cannotaffect any workload timing. This is where the array-versus-NDJSON contrastinside variant D is visible: same engine, same machine, same session, with fileframing as the only variable.

In [10]:
read_times = {}
for label, (comp_fn, fin_fn) in SOURCES.items():
    for side, fn in [("companies", comp_fn), ("financial_data", fin_fn)]:
        key = "%s | %s" % (label, side)
        _, times, err = timeit(lambda fn=fn: fn().count(), repeats=1)
        read_times[key] = {"times": times, "error": err}
        if err:
            print("%-44s FAILED  %s" % (key, err))
        else:
            print("%-44s %7.2fs" % (key, times[0]))

B: Spark + Mongo connector | companies         15.75s
B: Spark + Mongo connector | financial_data     5.05s
C: Spark + Parquet | companies                  1.48s
C: Spark + Parquet | financial_data             0.65s
D: Spark + JSON | companies                    36.73s
D: Spark + JSON | financial_data                4.00s


## Summary and persistence

In [11]:
print("%-8s %-32s %9s %9s %9s %9s" % ("workload", "variant", "median", "min", "max", "vs best"))
print("-" * 82)
for wname, wdata in WORKLOADS:
    completed = {l: t for l, (r, t, e) in wdata.items() if not e}
    if not completed:
        continue
    best = min(statistics.median(t) for t in completed.values())
    for label, (res, times, err) in wdata.items():
        if err:
            print("%-8s %-32s %9s" % (wname, label, "FAILED"))
            continue
        med = statistics.median(times)
        print("%-8s %-32s %8.2fs %8.2fs %8.2fs %8.2fx"
              % (wname, label, med, min(times), max(times), med / best))

summary = {
    "repeats": REPEATS,
    "selected_variants": SELECTED_VARIANTS,
    "read_only_timings_ran": "last",
    "cpu_cores": os.cpu_count(),
    "spark_master": spark.sparkContext.master,
    "driver_max_heap_gb": spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3,
    "spark_version": spark.version,
    "mongodb_version": db.command("buildInfo")["version"],
    "connector": CONNECTOR,
    "raw_json_bytes": os.path.getsize(RAW_COMPANIES),
    "collection_counts": COLLECTION_COUNTS,
    "indexes": INDEXES,
    "read_only_seconds": read_times,
    "workloads": {
        wname: {label: {"times": times, "error": err,
                        "rows": None if res is None else len(res)}
                for label, (res, times, err) in wdata.items()}
        for wname, wdata in WORKLOADS
    },
    "agreement": agreement,
    "formulation_effect": formulation,
    "engine_effect": engine,
    "float_tolerance": TOLERANCE,
}

# Per-selection filename so a partial run cannot overwrite a full one.
suffix = "" if len(SELECTED_VARIANTS) == 5 else "_" + "".join(SELECTED_VARIANTS)
out_path = os.path.join(DATA_DIR, "benchmark_results%s.json" % suffix)
with open(out_path, "w") as fh:
    json.dump(summary, fh, indent=2)
print("\nSaved to %s" % out_path)

# W4's aggregate is the input to the analysis chapter, so it is kept rather
# than discarded with the timings. Parquet is preferred as the source since it
# is the fastest variant; fall back to whichever completed.
w4_result = next((r for l, (r, t, e) in w4.items() if not e and l.startswith("C")), None)
if w4_result is None:
    w4_result = next((r for l, (r, t, e) in w4.items() if not e), None)
if w4_result:
    with open(os.path.join(DATA_DIR, "w4_industry_municipality.json"), "w") as fh:
        json.dump([{"naeringskode": k[0] or None, "kommunenummer": k[1] or None,
                    "antall": n, "sumDriftsinntekter": f[0], "sumDriftsresultat": f[1],
                    "sumEgenkapital": f[2], "sumGjeld": f[3]}
                   for k, n, f in w4_result], fh, indent=1)
    print("Saved %d groups to data/w4_industry_municipality.json" % len(w4_result))

workload variant                             median       min       max   vs best
----------------------------------------------------------------------------------
W1       A1: MongoDB (financial-first)       38.82s    38.73s    39.04s    14.79x
W1       A2: MongoDB (AS-first)               6.94s     6.81s     6.95s     2.64x
W1       B: Spark + Mongo connector           9.89s     9.24s    10.57s     3.77x
W1       C: Spark + Parquet                   2.62s     2.54s     2.64s     1.00x
W1       D: Spark + JSON                     36.95s    36.44s    37.70s    14.08x
W3       A: MongoDB aggregation               0.38s     0.36s     0.40s     1.00x
W3       B: Spark + Mongo connector           1.40s     1.37s     1.44s     3.70x
W3       C: Spark + Parquet                   1.55s     1.30s     1.62s     4.09x
W3       D: Spark + JSON                     34.15s    32.81s    34.64s    89.94x
W4       A1: MongoDB (financial-first)       18.74s    18.51s    18.75s     5.78x
W4       A2: Mo